# 🚀 Universal MLOps Orchestrator

Welcome to the frontend of the **Universal MLOps Platform**. This notebook provides a user-friendly graphical interface (GUI) to an advanced, fully config-driven command-line architecture. 

### 🌟 What this platform does:
This platform treats external machine learning repositories (like Music Source Separation models) as **black boxes**. You do not need to rewrite their code. 
1. **Idempotent Preparation:** Automatically clones external repos, robustly installs dependencies (using a line-by-line fallback and custom overrides), and downloads datasets.
2. **Execution, Retraining, & Tracking:** Injects your paths and hyperparameters dynamically into the external script. Supports resuming from checkpoints and logs everything to MLflow. Supports parallel multi-model execution.
3. **Zero-Touch Tuning:** Uses Ray Tune and Optuna to hunt for the best hyperparameters across multiple GPUs simultaneously.

### 📋 Workflow Overview:
1. **Setup:** Mount your Google Drive.
2. **Configure (Build):** Update `experiments.yaml` without touching code. Support for checkpoints included!
3. **Execute (Run):** Run the `prep` and `run` phases. You can batch multiple models to train in parallel or sequentially.
4. **Tune (Optimize):** Kick off a multi-GPU hyperparameter sweep.
5. **Dashboards:** View MLflow and Ray Tune/Optuna metrics in real-time.

## 🛠️ Step 1: Workspace & Drive Setup
Before the orchestrator can do anything, it needs access to your project files. This cell mounts your Google Drive to the Colab instance and navigates to the root of your MLOps platform.

**Example Usage:**
* If your platform folder is located at the root of your Google Drive, set the path to: `/content/drive/MyDrive/asm-wf-platform`
* Check the `mount_drive` box if you haven't connected your drive to this session yet.

In [ ]:
# @markdown ### Mount Drive & Set Project Root

mount_drive = False #@param {type:"boolean"}
project_path = "/content/drive/MyDrive/asm-wf-platform" #@param {type:"string"}

import os
from google.colab import drive

if mount_drive:
    drive.mount('/content/drive', force_remount=True)

# Create the directory if it doesn't exist, then move into it
os.makedirs(project_path, exist_ok=True)
os.chdir(project_path)

print(f"✅ Workspace active at: {os.getcwd()}")

## 📝 Step 2: The Experiment Builder (Config UI)
 `registry/experiments.yaml` UI, you can use this interface to inject a new experiment or overwrite an existing one. 

The orchestrator will automatically translate the `$DATASET_DIR` variable to the absolute path of your processed dataset.

**Example Usage:**
* **Experiment Name:** `exp_custom_roformer`
* **Base Command:** `python train.py --data_path $DATASET_DIR/train`
* **Extra Arguments:** `--model_type bs_roformer --batch_size 4 --epochs 100 --num_workers 2`

**Retraining / Resuming:** If your training crashed, or you want to fine-tune a pre-trained model, paste the path to the `.ckpt` or `.pt` file in `resume_checkpoint`. The orchestrator will automatically append `--start_check_point {path}` (or your model's equivalent flag) to the command!

In [ ]:
# @markdown ### Update `experiments.yaml`

import yaml
from pathlib import Path

experiment_name = "exp_custom_roformer" #@param {type:"string"}
model_registry_name = "bs_roformer" #@param ["bs_roformer", "mdx23c", "scnet"] {allow-input: true}
dataset_registry_name = "musdb18" #@param ["musdb18", "vocals_v1"] {allow-input: true}
mlflow_experiment_name = "stem_separation_baselines" #@param {type:"string"}

# @markdown ---
# @markdown ### Script Execution Arguments
base_command = "python train.py --data_path $DATASET_DIR/train" #@param {type:"string"}
extra_arguments = "--model_type bs_roformer --batch_size 4 --epochs 100 --num_workers 2" #@param {type:"string"}

# @markdown ---
# @markdown ### Retraining / Fine-Tuning
# Leave blank for a fresh run. If provided, it appends to extra_args.
resume_checkpoint = "" #@param {type:"string"}
# Change this flag to match whatever your specific train.py expects (e.g., --resume, --start_check_point, --load_model)
checkpoint_flag = "--start_check_point" #@param {type:"string"}

yaml_path = Path("core/registry/experiments.yaml")

# Safely load existing YAML
if yaml_path.exists():
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f) or {"experiments": {}}
else:
    data = {"experiments": {}}
    yaml_path.parent.mkdir(parents=True, exist_ok=True)

# Handle Checkpoint appending
final_extra_args = extra_arguments.strip()
if resume_checkpoint.strip():
    final_extra_args += f" {checkpoint_flag} {resume_checkpoint.strip()}"

# Update the specific experiment
data["experiments"][experiment_name] = {
    "model": model_registry_name,
    "dataset": dataset_registry_name,
    "mlflow_experiment": mlflow_experiment_name,
    "cmd": base_command,
    "extra_args": final_extra_args
}

# Save it back to disk safely
with open(yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False, sort_keys=False)

print(f"✅ Successfully saved '{experiment_name}' to {yaml_path}")
if resume_checkpoint.strip():
    print(f"🔄 Retraining enabled from: {resume_checkpoint}")

## 🎛️ Step: The Tuning Builder (Config UI)
Define the boundaries for Optuna and Ray Tune. This saves your search space directly into `tuning.yaml`.

In [ ]:
# @markdown ### Update `tuning.yaml`

import yaml
from pathlib import Path

tune_name = "tune_custom_roformer" #@param {type:"string"}
base_experiment = "exp_custom_roformer" #@param {type:"string"}

# @markdown ---
# @markdown ### Metric Tracking
target_metric = "val_loss" #@param {type:"string"}
mode = "min" #@param ["min", "max"]
metric_regex = "(?i)(?:valid(?:ation)?\\s*loss|val_loss)\\s*[:=]\\s*([0-9]+\\.[0-9]+)" #@param {type:"string"}

# @markdown ---
# @markdown ### Hardware & Search Space Limits
num_samples = 10 #@param {type:"integer"}
max_concurrent_trials = 2 #@param {type:"integer"}
max_epochs = 50 #@param {type:"integer"}

lr_min = 1.0e-5 #@param {type:"number"}
lr_max = 1.0e-3 #@param {type:"number"}
batch_size_choices = "2, 4" #@param {type:"string"}

yaml_path = Path("core/registry/tuning.yaml")

if yaml_path.exists():
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f) or {"tuning": {}}
else:
    data = {"tuning": {}}

# Parse batch sizes
bs_list = [int(x.strip()) for x in batch_size_choices.split(",") if x.strip().isdigit()]

data["tuning"][tune_name] = {
    "base_experiment": base_experiment,
    "target_metric": target_metric,
    "mode": mode,
    "metric_regex": metric_regex,
    "tune_config": {
        "num_samples": num_samples,
        "max_concurrent_trials": max_concurrent_trials
    },
    "asha_config": {
        "max_t": max_epochs,
        "grace_period": 3,
        "reduction_factor": 2
    },
    "search_space": {
        "lr": {"type": "loguniform", "low": lr_min, "high": lr_max},
        "batch_size": {"type": "choice", "values": bs_list}
    }
}

with open(yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False, sort_keys=False)

print(f"✅ Successfully saved '{tune_name}' search space to {yaml_path}")

## 🚀 Step 3: Orchestrator Execution Dashboard
trigger the `prep` and `run` phase of this pipeline. 

You can queue up multiple experiments to run back-to-back. The orchestrator will handle all Git cloning, pip installations (via pipreqs and robust fallbacks), data processing, and MLflow tracking.

**Example Usage:**
* **Target Experiments:** `exp_bs_roformer_mel, exp_mdx23c_vocals, exp_scnet_drums`
* **Run Preparation:** `True` (Check the box)
* **Prep Steps:** `all` (or specific phases like `clone install`)
* **Run Training:** `True` (Check the box)

**Parallel Execution:** If you provide multiple experiments (e.g., `exp_A, exp_B`) and check `run_in_parallel`, the UI will `prep` them sequentially (to avoid Git/Pip locks) and then `run` them simultaneously. Terminal output is saved to `logs/` to keep your Colab notebook clean!

In [ ]:
# @markdown ### Execute MLOps Pipeline

# @markdown **Target Experiments** *(comma-separated for multi-model runs)*
experiments_to_run = "exp_custom_roformer, exp_baseline" #@param {type:"string"}

# @markdown ---
# @markdown ### Execution Mode
run_in_parallel = True #@param {type:"boolean"}

# @markdown ---
# @markdown ### Phases to Execute
run_preparation = True #@param {type:"boolean"}
prep_steps = "all" #@param ["all", "clone", "install", "download", "process"] {allow-input: true}
run_training = True #@param {type:"boolean"}

import subprocess
import os
import time

experiment_list = [exp.strip() for exp in experiments_to_run.split(",") if exp.strip()]
os.makedirs("logs", exist_ok=True)

if not experiment_list:
    print("❌ No experiments defined.")
else:
    # --- STEP 1: SEQUENTIAL PREPARATION ---
    if run_preparation:
        print("\n" + "="*60)
        print(f"🛠️ PHASE 1: PREPARING ENVIRONMENTS SAFELY")
        print("="*60)
        for current_exp in experiment_list:
            prep_cmd = f"python core/orchestrator/orchestrator.py prep --exp {current_exp} --steps {prep_steps}"
            print(f"> Executing Prep for {current_exp}...")
            prep_code = subprocess.run(prep_cmd, shell=True).returncode
            if prep_code != 0:
                print(f"❌ Preparation failed for {current_exp}. It may fail in Phase 2.")

    # --- STEP 2: TRAINING EXECUTION ---
    if run_training:
        print("\n" + "="*60)
        print(f"🚀 PHASE 2: EXECUTING TRAINING ({'PARALLEL' if run_in_parallel else 'SEQUENTIAL'})")
        print("="*60)
        
        if run_in_parallel:
            # --- INFRASTRUCTURE CHECK ---
            try:
                import torch
                num_gpus = torch.cuda.device_count()
            except ImportError:
                num_gpus = 0
                
            print(f"🔍 INFRA CHECK: Detected {num_gpus} GPU(s).")
            if num_gpus == 0:
                print("⚠️ WARNING: No GPUs detected! Running in parallel on CPU will cause massive bottlenecks.")
                time.sleep(3)
            elif num_gpus < len(experiment_list):
                print(f"⚠️ WARNING: You are launching {len(experiment_list)} runs in parallel, but only have {num_gpus} GPU(s).")
                print("⚠️ Watch out for CUDA Out-Of-Memory (OOM) errors! Starting in 3 seconds...")
                time.sleep(3)
            else:
                print("✅ Sufficient hardware detected for parallel execution.")
            
            # Execute Parallel
            active_processes = []
            for idx, current_exp in enumerate(experiment_list):
                log_file_path = f"logs/{current_exp}_run.log"
                run_cmd = f"python core/orchestrator/orchestrator.py run --exp {current_exp}"
                
                print(f"> Launching {current_exp} in background...")
                print(f"  ↳ Output routed to: {log_file_path}")
                
                log_file = open(log_file_path, "w")
                process = subprocess.Popen(run_cmd, shell=True, stdout=log_file, stderr=subprocess.STDOUT)
                active_processes.append((current_exp, process, log_file))

            print("\n⏳ Waiting for all parallel training runs to complete...")
            for exp_name, proc, log_file in active_processes:
                proc.wait()
                log_file.close()
                if proc.returncode != 0:
                    print(f"❌ {exp_name} failed. Check logs/{exp_name}_run.log")
                else:
                    print(f"✅ {exp_name} completed successfully.")
                    
        else:
            # Execute Sequential
            for current_exp in experiment_list:
                run_cmd = f"python core/orchestrator/orchestrator.py run --exp {current_exp}"
                print(f"\n> Running {current_exp}...")
                subprocess.run(run_cmd, shell=True)

    print("\n🎉 ALL BATCH OPERATIONS FINISHED.")

## 🎛️ Step 4: Multi-GPU Hyperparameter Tuning
Launch the Ray Tune and Optuna sweep. This phase spins up multiple isolated subprocesses across your available GPUs. It automatically injects varying learning rates and batch sizes, parses the terminal output for validation loss, and kills underperforming trials early via ASHA.

**Example Usage:**
* Ensure your `core/registry/tuning.yaml` is fully configured.
* **Tuning Configuration:** `tune_bs_roformer_mel`
* If you run out of Colab GPU quota, check `force_cpu` to run a quick dry-run test without crashing.

In [ ]:
# @markdown ### Launch Ray Tune Sweep

tuning_configuration = "tune_custom_roformer" #@param {type:"string"}
force_cpu = False #@param {type:"boolean"}

import subprocess

print("\n" + "="*60)
print(f"🎯 STARTING HYPERPARAMETER TUNE: {tuning_configuration}")
print("="*60)

tune_cmd = f"python core/orchestrator/orchestrator.py tune --config {tuning_configuration}"

if force_cpu:
    tune_cmd = f"CUDA_VISIBLE_DEVICES='' {tune_cmd}"

subprocess.run(tune_cmd, shell=True)

## 📊 Step 5: Visualizations & Tracking Dashboards
Watch your training metrics and hyperparameter tuning sweeps in real-time.

* **Ray Tune / Optuna:** Natively outputs to TensorBoard. This cell loads an interactive TensorBoard right here in the notebook.
* **MLflow:** Because Colab blocks background ports, this cell uses `localtunnel` to give you a secure public URL to view your MLflow dashboard.

In [ ]:
# @markdown ### Start Dashboards

show_ray_tune_tensorboard = True #@param {type:"boolean"}
show_mlflow_ui = True #@param {type:"boolean"}

import os

if show_ray_tune_tensorboard:
    print("📈 Loading Ray Tune (Optuna) TensorBoard...")
    # Adjust this path if your tuning.yaml 'storage_path' is different
    ray_results_path = os.path.join(os.getcwd(), "ray_results")
    
    # Load tensorboard extension and display inline
    %load_ext tensorboard
    %tensorboard --logdir {ray_results_path}

if show_mlflow_ui:
    print("\n🧪 Launching MLflow UI via LocalTunnel...")
    # Run MLflow in the background on port 5000
    get_ipython().system_raw("mlflow ui --backend-store-uri mlruns --port 5000 &")
    
    print("👉 Click the 'loca.lt' link below to open MLflow!")
    print("⚠️  Note: If LocalTunnel asks for an 'Endpoint IP', it is usually the public IP of your Colab instance.")
    
    # Install localtunnel and expose port 5000
    !npm install -g localtunnel &> /dev/null
    !npx localtunnel --port 5000